In [ ]:
import tensorflow as tf
import keras
from keras.layers import Conv2DTranspose
print(tf.__version__)
print(keras.__version__)

In [ ]:
# Monkey patch removendo groups da assinatura original
original_init = Conv2DTranspose.__init__


def new_init(self, *args, **kwargs):
    kwargs.pop("groups", None)
    original_init(self, *args, **kwargs)


Conv2DTranspose.__init__ = new_init


In [ ]:
model = keras.models.load_model(
    r"D:\GeoPipe\data\000_models\trained_model1.h5",
    compile=False
)

print("Modelo carregado com sucesso!")

In [ ]:
model.save("D:\\GeoPipe\\data\\000_models\\trained_model1.keras")

In [ ]:

path_h5 = r"/media/weverton/TOSHIBA EXT1/GeoPipe/data/000_models/trained_model1.h5"
path_keras = (
    r"/media/weverton/TOSHIBA EXT1/GeoPipe/data/000_models/watnet.keras"
)

model = tf.keras.models.load_model(path_h5, compile=False)

model.save(path_keras)

In [ ]:
path_h5 = r"/media/weverton/TOSHIBA EXT1/GeoPipe/data/000_models/trained_model1.h5"

model = tf.keras.models.load_model(path_h5, compile=False)
print(f"Entradas: {[n.name for n in model.inputs]}")
print(f"Saídas: {[n.name for n in model.outputs]}")


In [ ]:
import torch

# 1. Carregar modelo e pesos (ajuste os nomes das classes/arquivos)
model = MyModelClass()
checkpoint = torch.load("seu_modelo.ckpt", map_location="cpu")
model.load_state_dict(checkpoint.get("state_dict", checkpoint))
model.eval()

# 2. Entrada de exemplo (use o shape padrão do seu modelo, ex: 224x224)
# O tamanho aqui serve apenas como "molde" para o grafo
dummy_input = torch.randn(1, 3, 224, 224)

# 3. Exportação com eixos dinâmicos (IGUAL AO KERAS)
torch.onnx.export(
    model,
    dummy_input,
    "modelo_pytorch.onnx",
    export_params=True,  # Armazena os pesos dentro do arquivo
    opset_version=11,  # Versão estável do ONNX
    do_constant_folding=True,  # Otimiza o modelo
    input_names=["input"],  # Nome da camada de entrada
    output_names=["output"],  # Nome da camada de saída
    # ESTA PARTE FAZ O FORMATO FICAR IGUAL AO KERAS:
    dynamic_axes={
        "input": {0: "batch_size"},  # Permite que o lote (batch) seja variável
        "output": {0: "batch_size"},
    },
)

print("Conversão concluída! O modelo agora aceita batch_size variável.")


In [1]:
import onnx


def check_batch_dim(model_path):
    # Carrega o modelo ONNX
    model = onnx.load(model_path)

    print(f"Verificando entradas do modelo: {model_path}\n")

    for input_tensor in model.graph.input:
        name = input_tensor.name
        shape = []

        # Itera sobre as dimensões do tensor de entrada
        for dim in input_tensor.type.tensor_type.shape.dim:
            if dim.HasField("dim_value"):
                # Dimensão fixa (ex: 224, 3)
                shape.append(str(dim.dim_value))
            elif dim.HasField("dim_param"):
                # Dimensão dinâmica (ex: 'batch_size', 'unk__0')
                shape.append(f"Dinâmica ('{dim.dim_param}')")
            else:
                shape.append("Indeterminada")

        print(f"Entrada: {name}")
        print(f"Shape: [{', '.join(shape)}]")

        # Analisa especificamente a primeira dimensão (Batch)
        first_dim = shape[0]
        if "Dinâmica" in first_dim:
            print("✅ Batch Dinâmico: Você pode enviar lotes de qualquer tamanho.")
        else:
            print(f"⚠️ Batch Fixo: O modelo espera exatamente {first_dim} amostra(s).")
        print("-" * 30)


# Execute o script passando o seu arquivo
check_batch_dim(
    "/media/weverton/TOSHIBA EXT1/GeoPipe/data/000_models/deepwatermap.onnx"
)


Verificando entradas do modelo: /media/weverton/TOSHIBA EXT1/GeoPipe/data/000_models/deepwatermap.onnx

Entrada: input
Shape: [Dinâmica ('unk__540'), Dinâmica ('unk__541'), Dinâmica ('unk__542'), 6]
✅ Batch Dinâmico: Você pode enviar lotes de qualquer tamanho.
------------------------------
